In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv("cubic_zirconia.csv")
pd.set_option("display.expand_frame",False)
data.shape

(26967, 11)

In [3]:
data.sample(10)

,Unnamed: 0,carat,cut,color,clarity,depth,table,x,y,z,price
15625,15626,1.59,Ideal,I,VS2,61.1,55.0,7.54,7.48,4.59,9972
21799,21800,0.91,Good,F,SI1,63.7,58.0,6.10,6.17,3.91,3993
19302,19303,0.52,Ideal,D,VS2,61.9,55.0,5.19,5.21,3.22,1802
9626,9627,0.71,Ideal,I,VS1,61.6,54.7,5.71,5.85,3.54,2397
10588,10589,1.50,Very Good,G,SI1,61.5,59.0,7.31,7.39,4.52,9645
25788,25789,0.37,Ideal,D,VS2,61.3,56.0,4.64,4.60,2.83,1124
4029,4030,0.56,Ideal,F,SI2,61.9,55.0,5.27,5.31,3.28,1212
2046,2047,0.32,Ideal,E,SI1,61.8,55.0,4.42,4.45,2.74,561
1352,1353,0.76,Ideal,G,VVS1,62.0,54.7,5.83,5.87,3.62,3671
20505,20506,4.01,Premium,I,I1,61.0,61.0,10.14,10.10,6.17,15223


In [ ]:
data["cut"].value_counts()

In [ ]:
data["color"].value_counts()

In [ ]:
data["clarity"].value_counts()

In [ ]:
data["clarity"]

In [ ]:
data.isnull().sum()

In [ ]:
data.describe()

In [ ]:
data.duplicated().sum()

In [ ]:
data.info()

In [ ]:
data = data.drop(columns=["Unnamed: 0"])

In [ ]:
data.shape

In [ ]:
import seaborn as sns
sns.displot(data["price"],kde = True) 

In [ ]:
sns.boxenplot(data["y"])

In [ ]:
data.corr(numeric_only= True)["price"].sort_values(ascending= True)

In [ ]:
value = data[["x","y","z"]]
for i in value:
    zero_count = (data[i]== 0).sum()
    print(f"{i}:",zero_count)

In [ ]:
from sklearn.model_selection import train_test_split
features = data.drop(columns= ["price"])
target = data["price"]

x_train, x_test,y_train, y_test = train_test_split(features,target,test_size=0.2,train_size=0.8,random_state= 42)


In [ ]:
x_train = x_train.replace(0,np.nan)
x_test = x_test.replace(0,np.nan)
x_train.isnull().sum()

In [ ]:
x_test.isnull().sum()

In [ ]:
x_test.describe()

In [ ]:
data.sample(5)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,RobustScaler
from sklearn.pipeline import Pipeline

ordinal_cols = ["cut","color","clarity"]
cut_categories = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_categories = ["J", "I", "H", "G", "F", "E", "D"]
clarity_categories = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

numaric_cols = [col for col in x_train.columns if col not in ordinal_cols]

preprocesser = ColumnTransformer(
    transformers= [
        (
         "ordinal_pipeline",
            Pipeline([
                ("encoder", OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])),
                ("scaler", RobustScaler())
            ]),
            ordinal_cols
         ),
         (
            "num_pipeline",
            Pipeline([
                ("imputer", SimpleImputer(strategy="mean")),
                ("scaler", RobustScaler())
            ]),
            numaric_cols
        )
    ],
    remainder="drop"
)


In [ ]:
print(numaric_cols)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components= 5,random_state= 42)

In [ ]:
from xgboost import XGBRegressor
model = XGBRegressor(
    n_estimators=100,
    learning_rate = 0.148,
    random_state = 42,
    max_depth = 5
)

In [ ]:
import optuna

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocesser",preprocesser),
    ("pca",pca),
    ("model",model)
])
pipeline.fit(x_train,y_train)
predicted_model = pipeline.predict(x_test)


In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

print("Mean Absolute Error:", mean_absolute_error(y_test, predicted_model))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, predicted_model))
print("R2 Score:", r2_score(y_test, predicted_model))

In [ ]:
train_pred = pipeline.predict(x_train)

from sklearn.metrics import r2_score

print("Train R2 :", r2_score(y_train, train_pred))
print("Test R2  :", r2_score(y_test, predicted_model))

In [ ]:
import joblib
joblib.dump(pipeline, 'gem_price_predictor.pkl')

In [ ]:
for i in range(1, 10):
    #Update the pipeline's PCA component
    pipeline.set_params(pca__n_components=i)

  # Fit and evaluate
    pipeline.fit(x_train, y_train)
    predicted_model = pipeline.predict(x_test)

    print(i, "R2 Score:", r2_score(y_test, predicted_model))